# Khon mask dense reconstruction (Colab GPU)

COLMAP's `patch_match_stereo` requires CUDA. The project is developed on an
Apple Silicon Mac, where the local build reports *"without CUDA"* and dense
stereo cannot run at all. This notebook is the one stage that runs elsewhere.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

Pipeline: upload the bundle from `scripts/02_dense_export.py`, undistort,
run patch-match stereo, fuse, download `fused.ply`.

Then back on your machine:
```
python scripts/03_dense_import.py ~/Downloads/fused.ply -s paths.run_id=<run>
```

## 1. Check the GPU

Stop here if this fails -- the rest cannot work without CUDA.

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    sys.exit(
        "No GPU detected. Set Runtime > Change runtime type > T4 GPU, then "
        "Runtime > Restart session and run this cell again."
    )
print(result.stdout.split("\n")[8] if len(result.stdout.split("\n")) > 8 else result.stdout)
print("GPU OK")

## 2. Install a CUDA-enabled COLMAP

**`apt install colmap` cannot work here.** COLMAP's own install guide states
that the distro packages are built without CUDA, so `patch_match_stereo`
refuses to run. We install the conda-forge build whose build string contains
`cuda` instead.

Takes 5-10 minutes. The next cell then *functionally* tests dense stereo
rather than trusting a version banner.

In [ ]:
%%bash
set -e
echo "driver:"; nvidia-smi | sed -n '3p'
if [ ! -x /content/bin/micromamba ]; then
  cd /content && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
if [ ! -x /content/mm/bin/colmap ]; then
  echo "installing colmap 4.1.1 CUDA + openimageio (a few GB, 5-10 min)..."
  # colmap is pinned to 4.1.1 to match the COLMAP that wrote the sparse model:
  # 4.x models carry rigs.bin/frames.bin that 3.x cannot read. Its CUDA builds
  # need a driver exposing CUDA >= 12.9 (Colab reports 13.0).
  #
  # openimageio is listed explicitly because EVERY conda-forge colmap 4.1.1
  # CUDA build omits it from its dependencies. Without it the binary dies with
  # 'libOpenImageIO.so.3.1: cannot open shared object file'.
  /content/bin/micromamba create -y -p /content/mm -c conda-forge \
      'colmap=4.1.1=cuda*' 'openimageio=3.1'
fi
echo done


In [ ]:
import os, subprocess, tempfile

os.environ['PATH'] = '/content/mm/bin:' + os.environ['PATH']

def sh(c):
    # errors='replace': COLMAP emits non-UTF-8 bytes and strict decoding raises.
    p = subprocess.run(c, shell=True, capture_output=True,
                       text=True, errors='replace')
    return (p.stdout or '') + (p.stderr or '')

# Three independent gates, all requiring POSITIVE evidence. An earlier version
# of this check only tested that the string 'requires CUDA' was absent, which a
# binary that cannot even load satisfies trivially -- it reported success on a
# COLMAP that was failing to start.
missing = [l.strip() for l in sh('ldd /content/mm/bin/colmap').splitlines() if 'not found' in l]
banner  = [l for l in sh('colmap --help').splitlines() if l.strip()]
runs    = any('COLMAP' in l.upper() for l in banner)
stereo  = sh(f'colmap patch_match_stereo --workspace_path {tempfile.mkdtemp()}')
cuda_ok = 'requires CUDA' not in stereo

print('1. no missing libraries :', 'PASS' if not missing else f'FAIL {missing}')
print('2. colmap actually runs :', 'PASS' if runs else 'FAIL')
print('  ', banner[0] if banner else '(no banner)')
print('3. CUDA dense stereo    :', 'PASS' if cuda_ok else 'FAIL')

assert not missing and runs and cuda_ok, 'NOT SAFE -- do not continue'
print('\nALL THREE PASS -- safe to continue')

## 3. Upload the bundle

Upload `dense_bundle_<run_id>.zip`, produced by `scripts/02_dense_export.py`.

For bundles over ~200 MB, mounting Drive is more reliable than the upload
widget -- use the commented alternative.

In [ ]:
import json, zipfile, pathlib
from google.colab import files

WORK = pathlib.Path("/content/work")
WORK.mkdir(exist_ok=True)

uploaded = files.upload()
bundle = pathlib.Path(list(uploaded)[0])

# Alternative for large bundles:
# from google.colab import drive; drive.mount('/content/drive')
# bundle = pathlib.Path('/content/drive/MyDrive/dense_bundle_mask01.zip')

with zipfile.ZipFile(bundle) as zf:
    zf.extractall(WORK)

settings = json.loads((WORK / "dense_settings.json").read_text())
n_images = len(list((WORK / "images").iterdir()))
print(f"run_id      : {settings['run_id']}")
print(f"images      : {n_images}")
print(f"masks       : {(WORK / 'masks').is_dir()}")
print(f"max size    : {settings['max_image_size']}")
settings

## 4a. Pre-flight: do the images match the camera model?

`image_undistorter` aborts with a bare C++ assertion when an image's size
disagrees with the camera recorded for it:

```
Check failed: distorted_camera.Width() == distorted_bitmap.Width() (4032 vs. 3024)
```

The usual cause is an EXIF rotation tag: COLMAP on macOS records the stored
size, COLMAP on Linux applies the rotation first. This cell detects that,
repairs it where it safely can, and otherwise explains exactly what to fix.

In [ ]:
!pip -q install pycolmap > /dev/null 2>&1
import pycolmap
from PIL import Image, ImageOps

rec = pycolmap.Reconstruction(str(WORK / 'sparse'))
mismatched, transposed, ok = [], [], 0

for image_id in rec.reg_image_ids():
    im = rec.image(image_id)
    cam = rec.camera(im.camera_id)
    path = WORK / 'images' / im.name
    if not path.exists():
        mismatched.append((im.name, 'file missing', ''))
        continue
    with Image.open(path) as raw:
        w, h = ImageOps.exif_transpose(raw).size   # what COLMAP-on-Linux sees
    if (w, h) == (cam.width, cam.height):
        ok += 1
    elif (h, w) == (cam.width, cam.height):
        transposed.append((im.name, path))
    else:
        mismatched.append((im.name, f'{w}x{h}', f'{cam.width}x{cam.height}'))

print(f'{ok} image(s) already match the camera model')

if transposed:
    print(f'{len(transposed)} image(s) are rotated relative to their camera -- repairing')
    for name, path in transposed:
        with Image.open(path) as raw:
            exif = raw.info.get('exif')
            fixed = ImageOps.exif_transpose(raw).rotate(-90, expand=True)
            fixed.save(path, 'JPEG', quality=97, subsampling=0,
                       **({'exif': exif} if exif else {}))
    print('   repaired -- rerun this cell to confirm before undistorting')

if mismatched:
    for name, got, want in mismatched[:5]:
        print(f'   {name}: image {got}, camera expects {want}')
    raise SystemExit(
        f'{len(mismatched)} image(s) cannot be reconciled with the sparse model. '
        'Re-ingest locally so orientation is normalised '
        '(scripts/00_prepare_images.py with --input pointing OUTSIDE the subject '
        'directory), re-run SfM, and export a fresh bundle.')

if not transposed:
    print('pre-flight OK -- safe to undistort')

## 4. Undistort

Rectifies the images and rewrites the model into the workspace layout that
patch-match stereo expects.

In [ ]:
import subprocess, time

DENSE = WORK / "dense"

def run(cmd):
    print("$", " ".join(str(c) for c in cmd))
    start = time.time()
    proc = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-3000:]); print(proc.stderr[-3000:])
        raise RuntimeError(f"failed: {cmd[1] if len(cmd) > 1 else cmd}")
    print(f"  done in {time.time() - start:.0f}s")
    return proc

run([
    "colmap", "image_undistorter",
    "--image_path", WORK / "images",
    "--input_path", WORK / "sparse",
    "--output_path", DENSE,
    "--output_type", "COLMAP",
    "--max_image_size", settings["max_image_size"],
])

## 5. Patch-match stereo (the GPU stage)

The slow step: roughly 15-40 minutes for 60-100 images on a T4.

`geom_consistency` cross-checks depth between views. It costs a second pass but
removes much of the noise that specular surfaces produce -- which on a gilded
Khon mask is the difference between a usable cloud and a spiky one.

In [ ]:
run([
    "colmap", "patch_match_stereo",
    "--workspace_path", DENSE,
    "--workspace_format", "COLMAP",
    "--PatchMatchStereo.geom_consistency",
    "true" if settings["geom_consistency"] else "false",
    "--PatchMatchStereo.window_radius", settings["window_radius"],
    "--PatchMatchStereo.num_samples", settings["num_samples"],
    "--PatchMatchStereo.max_image_size", settings["max_image_size"],
])

## 6. Fuse into a point cloud

`min_num_pixels` is the consistency requirement: a point must be seen and agreed
on by at least this many views. Raising it yields a cleaner but sparser cloud.

In [ ]:
FUSED = DENSE / "fused.ply"

run([
    "colmap", "stereo_fusion",
    "--workspace_path", DENSE,
    "--workspace_format", "COLMAP",
    "--input_type", "geometric" if settings["geom_consistency"] else "photometric",
    "--output_path", FUSED,
    "--StereoFusion.min_num_pixels", settings["fusion_min_num_pixels"],
    "--StereoFusion.max_reproj_error", settings["fusion_max_reproj_error"],
])

print(f"\nfused.ply: {FUSED.stat().st_size / 1e6:.1f} MB")

## 7. Sanity-check the result before downloading

In [ ]:
!pip -q install plyfile > /dev/null 2>&1
import numpy as np
from plyfile import PlyData

ply = PlyData.read(str(FUSED))
vertex = ply["vertex"]
points = np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1)

print(f"points        : {len(points):,}")
print(f"has colour    : {'red' in vertex.data.dtype.names}")
print(f"has normals   : {'nx' in vertex.data.dtype.names}")
print(f"bbox extent   : {(points.max(axis=0) - points.min(axis=0)).round(3)}")

assert len(points) > 1000, (
    "Very few points were fused. Usual causes: too little overlap between "
    "photos, or masks that removed nearly all of the object."
)
print("\nlooks reasonable -- download it in the next cell")

## 8. Download

Then, locally:
```
python scripts/03_dense_import.py ~/Downloads/fused.ply -s paths.run_id=<run_id>
python scripts/04_mesh.py -s paths.run_id=<run_id>
python scripts/05_texture.py -s paths.run_id=<run_id>
python scripts/06_evaluate.py -s paths.run_id=<run_id>
```

`03_dense_import.py` verifies the cloud lines up with the sparse model, so a
bundle downloaded from the wrong run is caught immediately.

In [ ]:
from google.colab import files
files.download(str(FUSED))

# The depth maps are worth keeping for the specularity study -- they show
# exactly where photometric matching failed on the gilded regions.
# !cd {DENSE} && zip -qr /content/depth_maps.zip stereo/depth_maps
# files.download('/content/depth_maps.zip')